In [ ]:
import json
with open('../chats/SillyTavern/Branch #133 - 2025-12-17@10h20m04s.jsonl', 'r') as json_file:
    json_list = list(json_file)
    
turns = []
current_prompt_parts = []
turn_id = 0

for json_str in json_list:
    msg = json.loads(json_str)
    text = msg["mes"].replace("\n\n", "\n").strip()

    if msg["name"] == "Miah":
        # Close a turn: pair Miah output with all prompts since last Miah
        turn_id += 1
        turns.append({
            "turn_id": turn_id,
            "prompt": "\n".join(current_prompt_parts).strip(),
            "miah": text
        })
        current_prompt_parts = []
    else:
        # Collect user/system prompts until next Miah response
        current_prompt_parts.append(text)

# Optional: if there are leftover prompts without response, keep them separately
dangling_prompts = "\n".join(current_prompt_parts).strip()

In [2]:
def group_turns(turns, turns_per_chunk=4):
    chunks = []
    for i in range(0, len(turns), turns_per_chunk):
        group = turns[i:i+turns_per_chunk]
        chunks.append(group)
    return chunks

turn_groups = group_turns(turns, turns_per_chunk=4)

In [3]:
from langchain_core.documents import Document

docs = []
for idx, group in enumerate(turn_groups, start=1):
    miah_text = "\n\n".join([t["miah"] for t in group]).strip()
    prompt_context = "\n\n".join(
        [f"[Turn {t['turn_id']} Prompt]\n{t['prompt']}" for t in group if t["prompt"]]
    ).strip()

    docs.append(Document(
        page_content=miah_text,
        metadata={
            "chunk_id": idx,
            "turn_ids": [t["turn_id"] for t in group],
            "prompt_context": prompt_context,  # kept for later inspection / traceability
        }
    ))


In [ ]:
#from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model_name = "nousresearch/hermes-3-llama-3.1-405b:free"
llm = model = init_chat_model(
    model=model_name,
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key="",
)
parser = StrOutputParser()

map_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are summarizing a creative roleplay/narrative session for future continuation.\n"
     "You will be given:\n"
     "A) Prompt context (what the user asked for)\n"
     "B) The character output (the story content)\n\n"
     "IMPORTANT:\n"
     "- Use the prompt context ONLY to interpret intent.\n"
     "- Do NOT summarize or include the prompt text as part of the memory.\n"
     "- Do NOT add turns info in the resume (like instructions of how to end the turns)\n"
     "- Summarize ONLY the narrative/story facts, character state, events, plans, relationships, tone.\n"
     "- Preserve names, places, unresolved threads, and decisions.\n"
    ),
    ("user",
     "PROMPT CONTEXT (do not summarize):\n{prompt_context}\n\n"
     "MIAH OUTPUT (summarize this):\n{miah_text}\n\n"
     "Write a memory chunk with sections:\n"
     "1) What happened (dense but readable)\n"
     "2) Character state (emotions, motivations, secrets)\n"
     "3) Open threads / hooks for next time\n"
     "4) Canon facts introduced (bullet list)\n"
    )
])

map_chain = map_prompt | llm | parser


In [8]:
from langchain_core.runnables.retry import RunnableRetry

map_chain_retry = map_chain.with_retry(
    retry_if_exception_type=(Exception,),
    stop_after_attempt=3,
    wait_exponential_jitter=True,
    exponential_jitter_params={"initial": 2},
)

In [ ]:
mapped_summaries = []
for d in docs:
    mapped = map_chain_retry.invoke({
        "prompt_context": d.metadata.get("prompt_context", ""),
        "miah_text": d.page_content,
    })
    mapped_summaries.append({
        "chunk_id": d.metadata["chunk_id"],
        "turn_ids": d.metadata["turn_ids"],
        "summary": mapped,
        "prompt_context": d.metadata.get("prompt_context", "")
    })


In [ ]:
mapped_summaries

In [ ]:
reduce_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You combine multiple memory chunks into a cohesive session summary for retrieval-augmented generation.\n"
     "Keep it detailed, consistent, and non-redundant. Do not mention prompts.\n"
     "Prefer canonical facts and actionable hooks for continuing the story."
    ),
    ("user",
     "Here are memory chunks from the session:\n\n{chunk_summaries}\n\n"
     "Produce the final session memory with:\n"
     "A) Narrative recap (multi-paragraph, not too short)\n"
     "B) Current character snapshot (goals, emotions, relationships, conflicts)\n"
     "C) World/canon facts (bullets)\n"
     "D) Continuation hooks (ranked list)\n"
     "E) Style/tone notes to maintain (bullets)\n"
    )
])

reduce_chain = reduce_prompt | llm | parser

final_memory = reduce_chain.invoke({
    "chunk_summaries": "\n\n---\n\n".join(
        [f"[Chunk {m['chunk_id']} | turns {m['turn_ids']}]\n{m['summary']}" for m in mapped_summaries]
    )
})


In [ ]:
final_memory

In [ ]:
memory_docs = [
    Document(
        page_content=m["summary"],
        metadata={
            "type": "memory_chunk",
            "chunk_id": m["chunk_id"],
            "turn_ids": m["turn_ids"],
            "prompt_context": m["prompt_context"],  # not part of embedding text
        }
    )
    for m in mapped_summaries
]

memory_docs.append(Document(
    page_content=final_memory,
    metadata={"type": "session_memory"}
))

import jsonlines
def save_docs_to_jsonl(documents_, file_path: str) -> None:
    with jsonlines.open(file_path, mode="w") as writer:
        for doc in documents_:
            writer.write(doc.model_dump())

save_docs_to_jsonl(memory_docs, f'summary_{model_name.replace("/","_").replace(":","_")}.jsonl')